# 03 · GraphConv / GIN
RDKit atom-graph → PyG. GCN vs GIN 멀티태스크 학습/평가 비교.

In [ ]:
# --- Colab 자가치유 셋업 (로컬/CI에서는 자동 스킵) ---
import os, sys, subprocess
try:
    import src  # repo 루트에서 실행 중이면 바로 import
except ModuleNotFoundError:
    if not os.path.exists("admet-property-predictor"):
        subprocess.run(["git", "clone",
            "https://github.com/zqvo04/admet-property-predictor.git"], check=False)
    os.chdir("admet-property-predictor")
    subprocess.run(["bash", "setup_colab.sh"], check=False)
    import src
print("src", src.__version__, "| cwd", os.getcwd())

In [ ]:
# === CONFIG (Colab에서 값만 키우면 전체 재현) ===
# 검증용 기본값은 작게 설정 — 실제 학습 시 EPOCHS↑, 엔드포인트 추가.
import os
TDC_PATH = "data/"
EPOCHS   = int(os.environ.get("ADMET_EPOCHS", 2))   # Colab 실전: 100
REG_EP   = ["Caco2"]          # 회귀 엔드포인트 (확장: Solubility, Lipophilicity, ...)
CLS_EP   = ["hERG"]           # 분류 엔드포인트 (확장: BBBP, CYP3A4_Inhibition, ...)
ENDPOINTS_USED = REG_EP + CLS_EP
SEED = 1
print("endpoints:", ENDPOINTS_USED, "| epochs:", EPOCHS)

In [ ]:
import numpy as np, pandas as pd, torch
from torch_geometric.loader import DataLoader as GDataLoader
from src.data import load_tdc_endpoint, build_multitask, ENDPOINTS
from src.featurizers import smiles_list_to_graphs
from src.models import build_model
from src.train import Trainer, TrainConfig, graph_dataset
from src.evaluate import evaluate_multitask, summarize
np.random.seed(SEED); torch.manual_seed(SEED)
data = {ep: load_tdc_endpoint(ep, seed=SEED, tdc_path=TDC_PATH) for ep in ENDPOINTS_USED}
task_names = list(data.keys()); task_types=[data[n].task_type for n in task_names]
type_vec=[0 if t=='reg' else 1 for t in task_types]

In [ ]:
def mt_loader(split, shuffle):
    mt = build_multitask(data, split)
    graphs, gmask = smiles_list_to_graphs(mt.smiles)
    Y, M = mt.Y[gmask], mt.mask[gmask]
    gset = graph_dataset(graphs, Y, M)
    return GDataLoader(gset, batch_size=64, shuffle=shuffle), mt, gmask
tl,_,_ = mt_loader('train', True)
vl,_,_ = mt_loader('valid', False)
tel, mt_te, te_mask = mt_loader('test', False)

## GCN vs GIN

In [ ]:
results = {}
for conv in ['gcn', 'gin']:
    model = build_model(conv, num_tasks=len(task_names))
    tr = Trainer(model, task_types=type_vec, cfg=TrainConfig(epochs=EPOCHS,
                 ckpt_dir='checkpoints/', run_name=f'{conv}_mt'))
    tr.fit(tl, vl)
    pred = tr.predict(tel)
    r = evaluate_multitask(pred, mt_te.Y[te_mask], mt_te.mask[te_mask],
                           task_names, task_types, mt_te.scalers)
    results[conv] = r
    print(conv.upper()); print(summarize(r, ENDPOINTS).to_string(index=False))

In [ ]:
rows = []
for conv, r in results.items():
    for ep, m in r.items():
        rows.append({'model': conv.upper(), 'endpoint': ep,
                     'official_metric': ENDPOINTS[ep]['metric'],
                     'value': m.get(ENDPOINTS[ep]['metric'])})
pd.DataFrame(rows)

✅ GCN vs GIN 비교 완료. 다음: `04_Comparison_Leaderboard`.